# AI Laboratory — Bayesian Networks and Autoregressive Language Models

## Part I — From Probability to Language

### Q1. Why is the chain-rule decomposition useful for generating text?

The chain rule decomposes the probability of a complete sequence into a
sequence of conditional probabilities. This allows a language model to
generate a sentence one token at a time.

Instead of calculating the probability of the entire sentence directly, the
model first chooses X1, then chooses X2 conditioned on the previous tokens,
then X3, and so on. Therefore, the chain-rule decomposition provides a
principled way to generate text sequentially.

## Part II - A Bayesian Network for Text
### Q2. What independence assumption is made?

The first-order model assumes that the current word depends only on the
immediately preceding word.

Therefore:

P(X_t | X_1, X_2, ..., X_{t-1}) ≈ P(X_t | X_{t-1})

In other words, once X_(t-1) is known, the earlier words do not provide
additional information for predicting X_t in this model.

## Part III: Build a Small Language Dataset
Use a small collection of sentences.
Following starting dataset used:
the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug
Convert all text to lower case.
For simplicity, initially treat each word as a token.
Add special tokens
⟨START⟩ ⟨END⟩
to mark the beginning and end of each sentence.

In [52]:
sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug"
]

tokenized_sentences = [sentence.lower().split() for sentence in sentences]

tokenized_sentences = [
    ["<START>"] + tokens + ["<END>"]
    for tokens in tokenized_sentences
]

for sentence in tokenized_sentences:
    print(sentence)

['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


## Part IV: Constructing the Conditional Probability Table

In [53]:
from collections import defaultdict

transition_counts = defaultdict(lambda: defaultdict(int))

for sentence in tokenized_sentences:
    for i in range(len(sentence) - 1):
        current_word = sentence[i]
        next_word = sentence[i + 1]
        transition_counts[current_word][next_word] += 1

for word, next_words in transition_counts.items():
    print(word, dict(next_words))

<START> {'the': 6}
the {'cat': 3, 'mat': 2, 'rug': 2, 'dog': 3, 'park': 2}
cat {'sat': 2, 'ran': 1}
sat {'on': 4}
on {'the': 4}
mat {'<END>': 2}
rug {'<END>': 2}
dog {'sat': 2, 'ran': 1}
ran {'to': 2}
to {'the': 2}
park {'<END>': 2}


In [54]:
# Convert transition counts into conditional probabilities
probabilities = {}

for word, next_words in transition_counts.items():
    total = sum(next_words.values())

    probabilities[word] = {
        next_word: count / total
        for next_word, count in next_words.items()
    }

for word, next_words in probabilities.items():
    print(f"\nP(next word | {word})")

    for next_word, probability in next_words.items():
        print(f"  {next_word}: {probability:.4f}")


P(next word | <START>)
  the: 1.0000

P(next word | the)
  cat: 0.2500
  mat: 0.1667
  rug: 0.1667
  dog: 0.2500
  park: 0.1667

P(next word | cat)
  sat: 0.6667
  ran: 0.3333

P(next word | sat)
  on: 1.0000

P(next word | on)
  the: 1.0000

P(next word | mat)
  <END>: 1.0000

P(next word | rug)
  <END>: 1.0000

P(next word | dog)
  sat: 0.6667
  ran: 0.3333

P(next word | ran)
  to: 1.0000

P(next word | to)
  the: 1.0000

P(next word | park)
  <END>: 1.0000


## Q3. Conditional Probability Tables

### P(next word | the)

P(cat | the) = 3/12 = 0.25
P(dog | the) = 3/12 = 0.25
P(mat | the) = 2/12 = 0.1667
P(rug | the) = 2/12 = 0.1667
P(park | the) = 2/12 = 0.1667

### P(next word | cat)

P(sat | cat) = 2/3 = 0.6667
P(ran | cat) = 1/3 = 0.3333

### P(next word | dog)

P(sat | dog) = 2/3 = 0.6667
P(ran | dog) = 1/3 = 0.3333

### P(next word | sat)

P(on | sat) = 1.0

### P(next word | ran)

P(to | ran) = 1.0

All other possible next-word transitions for these contexts have
probability zero because they were not observed in the training data.

## Part V — First-Order Autoregressive Language Model

We now implement the first-order autoregressive language model using
transition counts.

The model estimates:

P(X_t | X_{t-1})

and uses this conditional probability distribution to predict and generate
the next word.

In [55]:
import random
from collections import defaultdict

class FirstOrderLanguageModel:

    def __init__(self):
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.probabilities = {}

    def train(self, sentences):
        for sentence in sentences:
            for i in range(len(sentence) - 1):
                current_word = sentence[i]
                next_word = sentence[i + 1]

                self.transition_counts[current_word][next_word] += 1

        for current_word, next_words in self.transition_counts.items():
            total = sum(next_words.values())

            self.probabilities[current_word] = {
                next_word: count / total
                for next_word, count in next_words.items()
            }

    def show_probabilities(self, word):
        if word not in self.probabilities:
            print(f"No transitions observed for '{word}'")
            return

        print(f"P(next word | {word})")

        for next_word, probability in self.probabilities[word].items():
            print(f"{next_word}: {probability:.4f}")

    def predict_next(self, word):
        if word not in self.probabilities:
            return None

        return max(
            self.probabilities[word],
            key=self.probabilities[word].get
        )

    def generate_sentence(self):
        current_word = "<START>"
        generated = []

        while current_word != "<END>":
            generated.append(current_word)

            if current_word not in self.probabilities:
                break

            next_words = list(self.probabilities[current_word].keys())
            probabilities = list(self.probabilities[current_word].values())

            current_word = random.choices(
                next_words,
                weights=probabilities,
                k=1
            )[0]

        generated = generated[1:]

        if generated and generated[-1] == "<END>":
            generated = generated[:-1]

        return " ".join(generated)

In [56]:
model = FirstOrderLanguageModel()

model.train(tokenized_sentences)

print("Model trained successfully.")

Model trained successfully.


In [57]:
model.show_probabilities("the")

P(next word | the)
cat: 0.2500
mat: 0.1667
rug: 0.1667
dog: 0.2500
park: 0.1667


In [58]:
print("Most probable word after 'cat':", model.predict_next("cat"))
print("Most probable word after 'dog':", model.predict_next("dog"))
print("Most probable word after 'sat':", model.predict_next("sat"))

Most probable word after 'cat': sat
Most probable word after 'dog': sat
Most probable word after 'sat': on


In [59]:
for i in range(5):
    print(model.generate_sentence())

the mat
the park
the park
the dog sat on the dog sat on the cat sat on the cat sat on the cat sat on the dog sat on the rug
the dog sat on the dog sat on the park


## Part VI — Inspecting the Implementation

### Q4. Where are transition counts stored?

The transition counts are stored in the nested dictionary
`self.transition_counts`.

For each pair of consecutive words, the program increments the count using:

self.transition_counts[current_word][next_word] += 1

The outer key represents the current word and the inner key represents the
next word.

### Q5. Where is P(X_t | X_{t-1}) computed?

The conditional probabilities are computed in the `train()` method.

For each current word, the program calculates the total number of observed
transitions and divides each transition count by that total:

probability = count / total

Thus: P(next word | current word)
=
count(current word, next word)
/
total transitions from current word

### Q6. How does the program choose the next word?

The program supports two different methods.

The `predict_next()` function uses greedy selection. It chooses the word with
the maximum conditional probability using `max()`.

The `generate_sentence()` function uses probabilistic sampling with
`random.choices()`, using the conditional probabilities as weights.

Greedy selection always chooses the same most probable word for a given
context, whereas sampling can choose different words according to their
probabilities.

### Q7. What happens if the program encounters a word with no observed transition?

If the current word does not exist in the probability table, the program
cannot determine a next-word distribution.

In `show_probabilities()`, it prints:

"No transitions observed for ..."

In `predict_next()`, it returns `None`.

During generation, the program stops if there is no probability distribution
for the current word.

## Part VII — Test Probability Normalization

In [60]:
for word in model.probabilities:
    total = sum(model.probabilities[word].values())
    print(word, total)

<START> 1.0
the 1.0
cat 1.0
sat 1.0
on 1.0
mat 1.0
rug 1.0
dog 1.0
ran 1.0
to 1.0
park 1.0


In [61]:
for word in model.probabilities:
    total = sum(model.probabilities[word].values())

    if abs(total - 1.0) < 1e-9:
        print(f"{word}: PASS ({total:.6f})")
    else:
        print(f"{word}: FAIL ({total:.6f})")

<START>: PASS (1.000000)
the: PASS (1.000000)
cat: PASS (1.000000)
sat: PASS (1.000000)
on: PASS (1.000000)
mat: PASS (1.000000)
rug: PASS (1.000000)
dog: PASS (1.000000)
ran: PASS (1.000000)
to: PASS (1.000000)
park: PASS (1.000000)


### Q8. If one total is 0.87, what does this tell you?

If the probabilities for a particular current word sum to 0.87 instead of
approximately 1.0, the conditional probability distribution is not correctly
normalized.

This indicates an error in the probability calculation or implementation.
For a valid conditional probability distribution:

sum P(v | w) = 1

for every current word w.

## Part VIII — Next-Word Prediction

In [62]:
words_to_test = ["the", "cat", "dog", "sat", "ran"]

for word in words_to_test:
    model.show_probabilities(word)
    print()

P(next word | the)
cat: 0.2500
mat: 0.1667
rug: 0.1667
dog: 0.2500
park: 0.1667

P(next word | cat)
sat: 0.6667
ran: 0.3333

P(next word | dog)
sat: 0.6667
ran: 0.3333

P(next word | sat)
on: 1.0000

P(next word | ran)
to: 1.0000



In [63]:
for word in words_to_test:
    prediction = model.predict_next(word)
    print(f"After '{word}' -> '{prediction}'")

After 'the' -> 'cat'
After 'cat' -> 'sat'
After 'dog' -> 'sat'
After 'sat' -> 'on'
After 'ran' -> 'to'


### Q9. Are most-probable predictions always the same as personal expectation?

No. The most probable prediction is determined only by the training data and
the assumptions of the model.

A human can use broader context, linguistic knowledge, and meaning when
predicting the next word. The simple model does not have this knowledge and
only uses the conditional probabilities estimated from the small dataset.

Therefore, a probability model's prediction and a human's linguistic
expectation do not necessarily have to be the same.

## Part IX — Generate Text

In [64]:
generated_sentences = []

for i in range(20):
    sentence = model.generate_sentence()
    generated_sentences.append(sentence)

for i, sentence in enumerate(generated_sentences, start=1):
    print(f"{i}. {sentence}")

1. the dog ran to the cat ran to the dog sat on the mat
2. the cat sat on the park
3. the dog sat on the cat sat on the park
4. the park
5. the mat
6. the dog sat on the dog ran to the park
7. the cat sat on the dog sat on the cat sat on the cat ran to the mat
8. the rug
9. the rug
10. the mat
11. the mat
12. the rug
13. the park
14. the dog sat on the mat
15. the park
16. the dog sat on the cat sat on the dog ran to the park
17. the cat ran to the dog ran to the dog sat on the rug
18. the dog ran to the cat ran to the park
19. the dog sat on the cat ran to the rug
20. the mat


In [65]:
with open("generated_sentences.txt", "w") as file:
    for i, sentence in enumerate(generated_sentences, start=1):
        file.write(f"{i}. {sentence}\n")

print("Saved 20 generated sentences to generated_sentences.txt")

Saved 20 generated sentences to generated_sentences.txt


## Part X — Greedy vs Sampling

In [66]:
def generate_greedy(model, max_length=20):
    current_word = "<START>"
    generated = []

    for _ in range(max_length):
        if current_word not in model.probabilities:
            break

        next_word = model.predict_next(current_word)

        if next_word == "<END>":
            break

        generated.append(next_word)
        current_word = next_word

    return " ".join(generated)

In [67]:
greedy_sentences = []

for i in range(5):
    sentence = generate_greedy(model)
    greedy_sentences.append(sentence)

for i, sentence in enumerate(greedy_sentences, start=1):
    print(f"{i}. {sentence}")

1. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
2. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
3. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
4. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
5. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on


In [68]:
sampling_sentences = []

for i in range(5):
    sentence = model.generate_sentence()
    sampling_sentences.append(sentence)

for i, sentence in enumerate(sampling_sentences, start=1):
    print(f"{i}. {sentence}")

1. the dog sat on the rug
2. the dog sat on the dog sat on the cat ran to the dog sat on the cat ran to the rug
3. the dog ran to the mat
4. the park
5. the mat


### Q10. Greedy vs Sampling

Greedy generation produces less variation because it always selects the most
probable next word for a given context.

Sampling produces more variation because it randomly selects the next word
according to the learned probability distribution.

For example, if:

P(sat | cat) = 0.6667
P(ran | cat) = 0.3333

greedy generation will always choose "sat" after "cat", while sampling can
choose either "sat" or "ran", with "sat" being more likely.

Therefore, sampling generally produces greater variation than greedy
generation.

## Part XI — Second-Order Bayesian Network

## Q11. First-Order vs Second-Order Model

### 1. Graph structure

First-order:
X1 → X2 → X3 → X4

Each word depends only on the immediately preceding word.

Second-order:
X(t-2) → X(t) ← X(t-1)

The current word depends on the previous two words.

### 2. CPT

The first-order model uses:

P(X_t | X_(t-1))

The second-order model uses:

P(X_t | X_(t-2), X_(t-1))

Therefore, the second-order model has conditional probabilities for pairs
of previous words rather than individual previous words.

### 3. Context available

The first-order model uses one previous word as context.

The second-order model uses two previous words as context, so it can capture
some longer-range word relationships.

### 4. Amount of data required

The second-order model requires more training data because it must estimate
probabilities for combinations of two previous words. With limited data,
many such contexts may occur rarely or not at all, producing more
zero-probability transitions.


## Part XII — Second-Order Language Model

In [69]:
from collections import defaultdict

class SecondOrderLanguageModel:

    def __init__(self):
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.probabilities = {}
        self.start_counts = defaultdict(int)

    def train(self, sentences):
      for sentence in sentences:
        if len(sentence) > 1:
          self.start_counts[sentence[1]] += 1

      for sentence in sentences:
        for i in range(len(sentence) - 2):
          previous_word_1 = sentence[i]
          previous_word_2 = sentence[i + 1]
          next_word = sentence[i + 2]

          context = (previous_word_1, previous_word_2)

          self.transition_counts[context][next_word] += 1

      for context, next_words in self.transition_counts.items():
        total = sum(next_words.values())
        self.probabilities[context] = {
            next_word: count / total
            for next_word, count in next_words.items()
        }

    def show_probabilities(self, word1, word2):

        context = (word1, word2)

        if context not in self.probabilities:
            print(f"No transitions observed for {context}")
            return

        print(f"P(next word | {word1}, {word2})")

        for next_word, probability in self.probabilities[context].items():
            print(f"{next_word}: {probability:.4f}")

    def predict_next(self, word1, word2):

        context = (word1, word2)

        if context not in self.probabilities:
            return None

        return max(
            self.probabilities[context],
            key=self.probabilities[context].get
        )

    def generate_sentence(self, max_length=20):
      words = list(self.start_counts.keys())
      counts = list(self.start_counts.values())

      first_word = random.choices(
          words,
          weights=counts,
          k=1
      )[0]

      generated = [first_word]

      word1 = "<START>"
      word2 = first_word

      for _ in range(max_length):
        context = (word1, word2)

        if context not in self.probabilities:
            break

        next_words = list(self.probabilities[context].keys())
        probabilities = list(self.probabilities[context].values())

        next_word = random.choices(
            next_words,
            weights=probabilities,
            k=1
        )[0]

        if next_word == "<END>":
            break

        generated.append(next_word)

        word1 = word2
        word2 = next_word
      return " ".join(generated)

In [70]:
second_order_model = SecondOrderLanguageModel()

second_order_model.train(tokenized_sentences)

print("Second-order model trained successfully.")

Second-order model trained successfully.


In [71]:
second_order_model.show_probabilities("the", "cat")

P(next word | the, cat)
sat: 0.6667
ran: 0.3333


In [72]:
print(
    "Most probable word after 'the cat':",
    second_order_model.predict_next("the", "cat")
)

Most probable word after 'the cat': sat


In [73]:
for i in range(5):
    print(f"{i+1}. {second_order_model.generate_sentence()}")

1. the cat sat on the mat
2. the cat ran to the park
3. the cat ran to the park
4. the cat ran to the park
5. the dog sat on the rug


In [74]:
contexts_to_test = [
    ("the", "cat"),
    ("the", "dog"),
    ("cat", "sat"),
    ("dog", "sat"),
    ("on", "the")
]

for word1, word2 in contexts_to_test:
    second_order_model.show_probabilities(word1, word2)
    print()

P(next word | the, cat)
sat: 0.6667
ran: 0.3333

P(next word | the, dog)
sat: 0.6667
ran: 0.3333

P(next word | cat, sat)
on: 1.0000

P(next word | dog, sat)
on: 1.0000

P(next word | on, the)
mat: 0.5000
rug: 0.5000



## Part XIII: Compare First-Order and Second-Order Models
### Comparison of First-Order and Second-Order Models

The first-order model has 11 observed contexts and 17 probability entries.

The second-order model has 14 observed contexts and 18 probability entries.

The second-order model uses two previous words as context, so it can distinguish between different word sequences that may have the same immediately preceding word. This provides more contextual information for prediction.

However, the second-order model also requires more conditional probabilities to be estimated. With a small training dataset, some possible two-word contexts are not observed, leading to sparse probability distributions.

The generated sentences from both models were generally similar because the training dataset is very small. The second-order model can capture slightly more contextual relationships, while the first-order model is simpler and has fewer parameters.

In [75]:
# First-order model
first_order_contexts = len(model.probabilities)
first_order_parameters = sum(
    len(next_words) for next_words in model.probabilities.values()
)

# Second-order model
second_order_contexts = len(second_order_model.probabilities)
second_order_parameters = sum(
    len(next_words) for next_words in second_order_model.probabilities.values()
)

print("First-order model:")
print("Number of contexts:", first_order_contexts)
print("Number of probability entries:", first_order_parameters)

print("\nSecond-order model:")
print("Number of contexts:", second_order_contexts)
print("Number of probability entries:", second_order_parameters)

First-order model:
Number of contexts: 11
Number of probability entries: 17

Second-order model:
Number of contexts: 14
Number of probability entries: 18


In [76]:
second_order_sentences = []

for i in range(20):
    sentence = second_order_model.generate_sentence()
    second_order_sentences.append(sentence)

for i, sentence in enumerate(second_order_sentences, start=1):
    print(f"{i}. {sentence}")

1. the cat ran to the park
2. the cat ran to the park
3. the cat ran to the park
4. the cat sat on the rug
5. the dog sat on the rug
6. the dog sat on the mat
7. the cat ran to the park
8. the cat ran to the park
9. the cat ran to the park
10. the cat sat on the mat
11. the dog ran to the park
12. the dog ran to the park
13. the cat sat on the rug
14. the cat sat on the mat
15. the dog sat on the rug
16. the dog ran to the park
17. the cat ran to the park
18. the cat sat on the mat
19. the cat ran to the park
20. the cat sat on the rug


In [77]:
print("First-order unique sentences:",
      len(set(generated_sentences)))

print("Second-order unique sentences:",
      len(set(second_order_sentences)))

First-order unique sentences: 13
Second-order unique sentences: 6


In [78]:
vocab = set()

for sentence in tokenized_sentences:
    vocab.update(sentence)

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

print("Possible first-order context-next-word combinations:",
      vocab_size ** 2)

print("Possible second-order context-next-word combinations:",
      vocab_size ** 3)

Vocabulary size: 12
Possible first-order context-next-word combinations: 144
Possible second-order context-next-word combinations: 1728


## Q12. Why does more context improve prediction but make estimation harder?

The second-order model uses two previous words as context:

P(X_t | X_{t-2}, X_{t-1})

instead of the first-order model's:

P(X_t | X_{t-1})

Using more context can improve prediction because the model has more information about what word is likely to come next. For example, the pair of previous words can distinguish situations that would look identical when only the immediately preceding word is considered.

However, more context also increases the number of possible contexts and conditional probabilities that must be estimated. With limited training data, many combinations of two previous words may occur rarely or not occur at all. This results in sparse CPTs and more zero-probability or unseen contexts.

Therefore, increasing the context size can provide more information for prediction, but it also requires more training data to estimate the conditional probabilities reliably.

## Part XIV. Modern Language Models

The simple language models implemented in this lab estimate conditional
probabilities directly from word-transition counts.

The first-order model estimates:

P(X_t | X_{t-1})

while the second-order model estimates:

P(X_t | X_{t-2}, X_{t-1})

Modern language models also use autoregressive probability modelling, where
the probability of the next token is conditioned on previous tokens.
However, instead of using simple frequency-based CPTs, modern models use
neural networks to learn much richer representations and conditional
probability distributions.

The simple models in this lab therefore illustrate the basic probabilistic
structure behind language generation. Modern language models use much more
powerful methods to estimate these probabilities and can capture much more
complex relationships in language.

## Q13. Reflection on LLM Use and Validation

Approach B is preferable because it specifies the intended probabilistic
model before asking the LLM to implement it.

First, specifying the intended behaviour makes clear what the program is
supposed to do. Second, understanding the representation ensures that the
implementation corresponds to the intended probabilistic model.

The generated implementation must then be validated by inspecting the code
and testing probabilistic invariants, such as checking that all conditional
probability distributions sum to 1.

It is also important to distinguish the implementation from the model. The
model is the probabilistic specification, while the Python program is an
implementation of that specification.

In this lab, the LLM-generated code was inspected and tested rather than
being accepted without verification.

## Q14. Final Reflection

This lab demonstrated how probability, Bayesian networks, and autoregressive
language models are connected.

The chain rule provides a way to factorize the joint probability of a
sequence into conditional probabilities. A first-order model simplifies this
by assuming that the next word depends only on the previous word, while a
second-order model uses the previous two words.

The experiment showed that increasing the context provides more information
for prediction, but also increases the number of conditional probabilities
that must be estimated. With a small dataset, this can result in sparse
probability tables and unseen contexts.

The LLM-assisted implementation was useful for converting the probabilistic
design into Python code, but testing and inspection were necessary to verify
that the implementation actually matched the intended model. Therefore,
understanding the probabilistic structure was important even when an LLM was
used to assist with programming.